<a href="https://colab.research.google.com/github/dodo701/goyu-ai/blob/main/goyu-general/scripts/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GOYU General - KoAlpaca Training

This notebook trains the GOYU General model using a sliced version of the `beomi/KoAlpaca-v1.1a` dataset.

In [1]:
!pip install datasets sentencepiece torch

In [2]:
!git clone https://github.com/dodo701/goyu-ai.git
%cd goyu-ai/goyu-general

Cloning into 'goyu-ai'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 103 (delta 13), reused 97 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 5.07 MiB | 22.67 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/goyu-ai/goyu-general


## 1. Load Dataset and Prepare Tokenizer

데이터세트 크기 설정.

In [3]:
dataset_size=int(input())

21150


데이터세트 로드

In [4]:
from datasets import load_dataset

# Load KoAlpaca dataset
dataset = load_dataset('beomi/KoAlpaca-v1.1a', split='train')

# Slice the first 400 examples
dataset = dataset.select(range(dataset_size))

# Format data into a text file for tokenizer training
temp_text_file = 'temp_corpus.txt'
with open(temp_text_file, 'w', encoding='utf-8') as f:
    for row in dataset:
        # We use the prompt format: User: {instruction}\nAssistant: {output}
        f.write(f"User: {row['instruction']}\n")
        f.write(f"Assistant: {row['output']}\n")


README.md:   0%|          | 0.00/1.75k [00:00<?, ?B/s]

data/train-00000-of-00001-21df739eb88d71(…): reconstructing file:   0%|          |  0.00B / 12.9MB            

data/train-00000-of-00001-21df739eb88d71(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/21155 [00:00<?, ? examples/s]

In [5]:
from src.tokenizer import train_tokenizer, GoyuTokenizer
import os

# Train tokenizer
os.makedirs('data/processed', exist_ok=True)
tokenizer_prefix = 'data/processed/tokenizer'
train_tokenizer(temp_text_file, tokenizer_prefix, vocab_size=32000) # smaller vocab for 400 items

tokenizer = GoyuTokenizer(f'{tokenizer_prefix}.model')
print(f"Tokenizer trained. Vocab size: {tokenizer.vocab_size}")


Tokenizer trained. Vocab size: 32000


## 2. Prepare Data for Model

배치 크기 설정.

In [10]:
set_batch_size=int(input("배치 크기 설정."))
set_epoch_size=int(input("에포크 크기 설정."))

배치 크기 설정.8
에포크 크기 설정.3


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
import json

class KoAlpacaDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=512):
        self.data = []
        for row in hf_dataset:
            instruction = row['instruction']
            output = row['output']

            # Prompt format
            prompt = f"User: {instruction}\nAssistant: "
            full_text = prompt + output

            # Encode
            full_ids = tokenizer.encode(full_text, add_bos=True, add_eos=True)
            prompt_ids = tokenizer.encode(prompt, add_bos=True, add_eos=False)

            if len(full_ids) > max_length:
                full_ids = full_ids[:max_length]

            # Create labels (ignore prompt for loss calculation by setting to -100)
            labels = full_ids.copy()
            labels[:len(prompt_ids)] = [-100] * min(len(prompt_ids), len(labels))

            self.data.append({
                'input_ids': torch.tensor(full_ids, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long)
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    labels = [item['labels'] for item in batch]

    # Pad sequences
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_id)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    return input_ids, labels

train_dataset = KoAlpacaDataset(dataset, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=set_batch_size, shuffle=True, collate_fn=collate_fn)


In [26]:
# Take one batch to inspect the training samples
batch = next(iter(train_loader))
input_ids_batch, labels_batch = batch

print(f"Batch Input Shape: {input_ids_batch.shape}")
print(f"Batch Label Shape: {labels_batch.shape}\n")

# Inspect the first 3 samples in the batch
for i in range(min(3, len(input_ids_batch))):
    print(f"=== Sample {i+1} ===")
    input_ids = input_ids_batch[i].tolist()
    labels = labels_batch[i].tolist()

    # Filter out padding token from inputs
    clean_input_ids = [tid for tid in input_ids if tid != tokenizer.pad_id]
    decoded_input = tokenizer.decode(clean_input_ids)

    # Find labels that are not -100 (which are the actual target assistant tokens we calculate loss on)
    target_ids = [tid for tid in labels if tid != -100]
    decoded_target = tokenizer.decode(target_ids)

    print("[Decoded Input Sequence]")
    print(decoded_input)
    print("\n[Decoded Target Assistant Response (Loss calculated on this)]")
    print(decoded_target)
    print("="*50 + "\n")

Batch Input Shape: torch.Size([8, 245])
Batch Label Shape: torch.Size([8, 245])

=== Sample 1 ===
[Decoded Input Sequence]
User: A3 + B3 = 1729, C3 + D3 = 1729일 때, A + B + C + D의 값은 얼마인가요? Assistant: 수학자 라마누잔의 일화를 기반으로 한 사기성이 짙은 풀이가 있습니다. A3 + B3 = 1729은 13 + 123로, C3 + D3 = 1729은 93 + 103로 표현이 가능합니다. 따라서, A + B + C + D = 1 + 9 + 10 + 12 = 32입니다. 하지만, 이 풀이 외에 좀 더 정석적인 방법은 없습니다.

[Decoded Target Assistant Response (Loss calculated on this)]
수학자 라마누잔의 일화를 기반으로 한 사기성이 짙은 풀이가 있습니다. A3 + B3 = 1729은 13 + 123로, C3 + D3 = 1729은 93 + 103로 표현이 가능합니다. 따라서, A + B + C + D = 1 + 9 + 10 + 12 = 32입니다. 하지만, 이 풀이 외에 좀 더 정석적인 방법은 없습니다.

=== Sample 2 ===
[Decoded Input Sequence]
User: 우리나라 최초의 신용카드는 무엇이며, 현재의 카드와 어떤 차이점이 있을까요? Assistant: 우리나라 최초의 신용카드는 1969년에 나온 '신세계백화점카드'입니다. 이는 이 회사와 관련된 일부 사람에게만 소량 발급됐고, 사용처도 신세계백화점에 국한됐기 때문에 현재 사용되는 신용카드와는 차이가 있습니다. 그러나 카드 소지자에게 외상으로 물건.서비스를 팔고 나중에 돈을 받았던 만큼 국내 최초의 신용카드로 인정받습니다. 금융 산업의 발전과 함께, 현재 신용카드는 전자적인 결제 기능을 중심으로 나머지 정보들이 칩 안에 저장되고 있습니다. 또한 최근에는 Apple Pay나 Samsu

## 3. Train Model

In [11]:
from src.model import GoyuGeneral, ModelConfig
import torch.optim as optim
import time
import inspect

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('config/base.json', 'r') as f:
    config_dict = json.load(f)

config_dict['vocab_size'] = tokenizer.vocab_size

# Filter out arguments that are not accepted by ModelConfig.__init__
sig = inspect.signature(ModelConfig.__init__)
valid_keys = [p for p in sig.parameters.keys() if p != 'self']
filtered_config_dict = {k: v for k, v in config_dict.items() if k in valid_keys}

config = ModelConfig(**filtered_config_dict)

model = GoyuGeneral(config).to(device)

optimizer = optim.AdamW(model.parameters(), lr=config_dict.get('learning_rate', 3e-4))

epoc_steps = 3
epochs = set_epoch_size
model.train()

for epoch in range(epochs):
    total_loss = 0
    start_time = time.time()
    for batch_idx, (input_ids, labels) in enumerate(train_loader):
        input_ids, labels = input_ids.to(device), labels.to(device)

        optimizer.zero_grad()
        logits, loss = model(input_ids, labels=labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f'Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1} completed in {time.time()-start_time:.2f}s | Average Loss: {avg_loss:.4f}\n')

Epoch 1/3 | Batch 0/2644 | Loss: 10.0521
Epoch 1/3 | Batch 50/2644 | Loss: 5.2212
Epoch 1/3 | Batch 100/2644 | Loss: 3.3111
Epoch 1/3 | Batch 150/2644 | Loss: 2.2457
Epoch 1/3 | Batch 200/2644 | Loss: 1.4584
Epoch 1/3 | Batch 250/2644 | Loss: 1.0502
Epoch 1/3 | Batch 300/2644 | Loss: 0.7146
Epoch 1/3 | Batch 350/2644 | Loss: 0.5191
Epoch 1/3 | Batch 400/2644 | Loss: 0.3924
Epoch 1/3 | Batch 450/2644 | Loss: 0.3214
Epoch 1/3 | Batch 500/2644 | Loss: 0.2558
Epoch 1/3 | Batch 550/2644 | Loss: 0.2853
Epoch 1/3 | Batch 600/2644 | Loss: 0.2062
Epoch 1/3 | Batch 650/2644 | Loss: 0.1497
Epoch 1/3 | Batch 700/2644 | Loss: 0.1428
Epoch 1/3 | Batch 750/2644 | Loss: 0.1137
Epoch 1/3 | Batch 800/2644 | Loss: 0.0771
Epoch 1/3 | Batch 850/2644 | Loss: 0.0832
Epoch 1/3 | Batch 900/2644 | Loss: 0.0756
Epoch 1/3 | Batch 950/2644 | Loss: 0.0818
Epoch 1/3 | Batch 1000/2644 | Loss: 0.0469
Epoch 1/3 | Batch 1050/2644 | Loss: 0.0881
Epoch 1/3 | Batch 1100/2644 | Loss: 0.0480
Epoch 1/3 | Batch 1150/2644 | Los

## 4. Save Model

In [13]:
import os

os.makedirs('models', exist_ok=True)
save_path = 'models/koalpaca_model.pt'

# Calculate total number of parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

torch.save({
    'model': model.state_dict(),
    'config': config_dict
}, save_path)

# Calculate file size
file_size_bytes = os.path.getsize(save_path)
file_size_mb = file_size_bytes / (1024 * 1024)

print(f"Model saved to {save_path}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Saved file size: {file_size_mb:.2f} MB")

Model saved to models/koalpaca_model.pt
Total parameters: 26,450,304
Trainable parameters: 26,450,304
Saved file size: 100.92 MB


## 5. Inference and Download

In [23]:
import torch.nn.functional as F

model.eval()
test_instruction = "스웨터의 유래는 어디에서 시작되었나요?"
prompt = f"User: {test_instruction}\nAssistant: "

input_ids = tokenizer.encode(prompt, add_bos=True, add_eos=False)
x = torch.tensor([input_ids], dtype=torch.long, device=device)

max_new_tokens = 100
temperature = 0.1
top_k = 50
top_p = 0.9
repetition_penalty = 1.0

with torch.no_grad():
    for _ in range(max_new_tokens):
        x_cond = x[:, -model.config.block_size:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :]

        # Apply repetition penalty
        for token_id in set(x[0].tolist()):
            if logits[0, token_id] > 0:
                logits[0, token_id] /= repetition_penalty
            else:
                logits[0, token_id] *= repetition_penalty

        # Apply temperature
        logits = logits / temperature

        # Apply Top-K filtering
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

        # Apply Top-p (nucleus) filtering
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            # Remove tokens with cumulative probability above the threshold
            sorted_indices_to_remove = cumulative_probs > top_p
            # Shift the indices to the right to keep the first token above the threshold
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            for i in range(logits.size(0)):
                indices_to_remove = sorted_indices[i][sorted_indices_to_remove[i]]
                logits[i, indices_to_remove] = -float('Inf')

        # Sample next token
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        x = torch.cat((x, next_id), dim=1)
        if next_id.item() == tokenizer.eos_id:
            break

generated_ids = x[0].tolist()[len(input_ids):]
response = tokenizer.decode(generated_ids)
print("Prompt:", prompt)
print("Response:", response)

Prompt: User: 스웨터의 유래는 어디에서 시작되었나요?
Assistant: 
Response: ::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::


In [19]:
from google.colab import files
files.download(save_path)
# To download the tokenizer as well:
files.download('data/processed/tokenizer.model')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>